# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [37]:
import importlib.util
import sys
import os
from dotenv import load_dotenv
load_dotenv()

# Fix sqlite issue in Udacity workspace
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from pydantic import BaseModel
from typing import List
from openai import OpenAI


In [38]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)


In [39]:


class EvaluationReport(BaseModel):
    useful: bool
    description: str


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [40]:
def retrieve_game(query: str):

    chroma_client = chromadb.PersistentClient(path="chromadb")

    embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=OPENAI_API_KEY,
        model_name="text-embedding-3-small"
    )

    collection = chroma_client.get_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )

    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    return results["documents"][0]


#### Evaluate Retrieval Tool

In [41]:
def evaluate_retrieval(question: str, retrieved_docs: List[str]):

    prompt = f"""
    Evaluate whether the retrieved documents are sufficient
    to answer the user question.

    Question:
    {question}

    Retrieved Documents:
    {retrieved_docs}

    Return JSON:
    {{
        "useful": true/false,
        "description": "short explanation"
    }}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    import json
    data = json.loads(response.choices[0].message.content)
    return EvaluationReport(**data)


#### Game Web Search Tool

In [42]:
def game_web_search(question: str):

    tavily = TavilyClient(api_key=TAVILY_API_KEY)

    response = tavily.search(
        query=question,
        search_depth="advanced",
        max_results=3
    )

    return response


### Agent

In [43]:
class Agent:

    def __init__(self, model_name, tools=None, instructions=""):

        self.model_name = model_name
        self.tools = {t.__name__: t for t in tools}
        self.instructions = instructions

        # Multi-session memory
        self.sessions = {}

        # Long-term memory (optional advanced feature)
        self.long_term_memory = []

    def save_to_memory(self, session_id, question, answer):

        self.long_term_memory.append({
            "session_id": session_id,
            "question": question,
            "answer": answer
        })

    def invoke(self, query, session_id="default_session"):

        # Create new session if needed
        if session_id not in self.sessions:
            self.sessions[session_id] = []

        # Store user message in session history
        self.sessions[session_id].append({
            "role": "user",
            "content": query
        })

        # -------------------
        # STATE 1: Retrieve
        # -------------------
        retrieved_docs = self.tools["retrieve_game"](query)

        # -------------------
        # STATE 2: Evaluate
        # -------------------
        evaluation = self.tools["evaluate_retrieval"](query, retrieved_docs)

        # -------------------
        # STATE 3: Decide
        # -------------------
        if evaluation.useful:

            context = f"""
            Use the retrieved documents to answer clearly and cite sources.

            Question:
            {query}

            Retrieved Documents:
            {retrieved_docs}
            """

        else:

            web_results = self.tools["game_web_search"](query)

            context = f"""
            The local database was insufficient.

            Use the web search results to answer clearly and cite sources.

            Question:
            {query}

            Web Results:
            {web_results}
            """

        # -------------------
        # TRUE STATEFUL CALL
        # -------------------

        messages = [{"role": "system", "content": self.instructions}]

        # Add previous conversation history
        messages.extend(self.sessions[session_id])

        # Add current RAG/web context as user message
        messages.append({
            "role": "user",
            "content": context
        })

        response = client.chat.completions.create(
            model=self.model_name,
            messages=messages
        )

        answer = response.choices[0].message.content

        # Store assistant response
        self.sessions[session_id].append({
            "role": "assistant",
            "content": answer
        })

        # Save to long-term memory
        self.save_to_memory(session_id, query, answer)

        return answer

    def get_session_messages(self, session_id):
        return self.sessions.get(session_id, [])


In [44]:
instructions = """
You are UdaPlay, an AI research agent for the video game industry.

Workflow:
1. Retrieve documents from vector database.
2. Evaluate retrieval usefulness.
3. If insufficient, perform web search.
4. Provide structured and cited answer.
5. Maintain session context via session_id.
"""
agent = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=instructions
)


In [45]:
session_1 = "video_history"

questions_1 = [
    "When was Pokémon Gold and Silver released?",
    "Which was the first 3D Mario platformer?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

for q in questions_1:
    print("\nQUESTION:", q)
    print("ANSWER:", agent.invoke(q, session_id=session_1))



QUESTION: When was Pokémon Gold and Silver released?
ANSWER: Pokémon Gold and Silver were released in Japan on November 21, 1999, for the Game Boy Color. These games introduced a new region, new Pokémon, and various gameplay mechanics as part of the second generation of Pokémon games ([Game Boy Color] Pokémon Gold and Silver (1999)).

QUESTION: Which was the first 3D Mario platformer?
ANSWER: The first 3D Mario platformer is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This game was groundbreaking, setting new standards for the platforming genre and featuring Mario's quest to rescue Princess Peach ([Nintendo 64] Super Mario 64 (1996)).

QUESTION: Was Mortal Kombat X released for PlayStation 5?
ANSWER: **Mortal Kombat X** was not released specifically for the PlayStation 5, but it is playable on the PS5 through backward compatibility. Players may need to update their PS5 system to the latest software for optimal performance, although some features from the PS4 ve

In [46]:
session_2 = "competitive_games"

print(agent.invoke(
    "What year was League of Legends released?",
    session_id=session_2
))


League of Legends was officially released on **October 27, 2009**. The game, developed by Riot Games, had been in closed beta testing prior to its launch, starting in April 2009. This popular multiplayer online battle arena (MOBA) game quickly gained traction within the gaming community and has since become a staple in the esports scene ([League of Legends Wiki](https://leagueoflegends.fandom.com/wiki/League_of_Legends), [The Spike](https://www.thespike.gg/league-of-legends/beginner-guides/league-of-legends-release-date), [Medium](https://williamlim3.medium.com/legendary-the-story-of-league-of-legends-from-2009-2018-68c578a0563f)).


In [47]:
print("\n--- SESSION 1 HISTORY ---")
for msg in agent.get_session_messages(session_1):
    print(msg["role"], ":", msg["content"])

print("\n--- SESSION 2 HISTORY ---")
for msg in agent.get_session_messages(session_2):
    print(msg["role"], ":", msg["content"])



--- SESSION 1 HISTORY ---
user : When was Pokémon Gold and Silver released?
assistant : Pokémon Gold and Silver were released in Japan on November 21, 1999, for the Game Boy Color. These games introduced a new region, new Pokémon, and various gameplay mechanics as part of the second generation of Pokémon games ([Game Boy Color] Pokémon Gold and Silver (1999)).
user : Which was the first 3D Mario platformer?
assistant : The first 3D Mario platformer is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This game was groundbreaking, setting new standards for the platforming genre and featuring Mario's quest to rescue Princess Peach ([Nintendo 64] Super Mario 64 (1996)).
user : Was Mortal Kombat X released for PlayStation 5?
assistant : **Mortal Kombat X** was not released specifically for the PlayStation 5, but it is playable on the PS5 through backward compatibility. Players may need to update their PS5 system to the latest software for optimal performance, although so

### (Optional) Advanced

In [48]:
print("\n--- LONG TERM MEMORY ---")
print(agent.long_term_memory)


--- LONG TERM MEMORY ---
[{'session_id': 'video_history', 'question': 'When was Pokémon Gold and Silver released?', 'answer': 'Pokémon Gold and Silver were released in Japan on November 21, 1999, for the Game Boy Color. These games introduced a new region, new Pokémon, and various gameplay mechanics as part of the second generation of Pokémon games ([Game Boy Color] Pokémon Gold and Silver (1999)).'}, {'session_id': 'video_history', 'question': 'Which was the first 3D Mario platformer?', 'answer': "The first 3D Mario platformer is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This game was groundbreaking, setting new standards for the platforming genre and featuring Mario's quest to rescue Princess Peach ([Nintendo 64] Super Mario 64 (1996))."}, {'session_id': 'video_history', 'question': 'Was Mortal Kombat X released for PlayStation 5?', 'answer': '**Mortal Kombat X** was not released specifically for the PlayStation 5, but it is playable on the PS5 through back